# Fine-tuning T5 Tiếng Việt bằng QLoRA/LoRA trên Google Colab phục vụ chuyển các cấu trúc phức thành các fact
Model: **ViT5 Base**  
Framework: Hugging Face Transformers + TRL + PEFT

> Trước khi chạy: vào **Runtime -> Change runtime type -> GPU** (chọn T4 hoặc cao hơn).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd '/content/drive/MyDrive/Project/Xet_Khen_Thuong/Fine-Tune-Vi-T5-Tach-Cau'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1zfAm_7GTOZncmyIUQiyFVmps4BX8Te2j/Xet_Khen_Thuong/Fine-Tune-Vi-T5-Tach-Cau


In [2]:
!pip install -q -U \
    transformers==4.57.6\
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    sentencepiece



In [3]:
import transformers
import peft
import torch

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("torch:", torch.__version__)

transformers: 4.57.6
peft: 0.21.2
torch: 2.14.1+cu130


Nếu Colab yêu cầu **restart runtime** sau khi cài, hãy chọn `Runtime -> Restart session` rồi chạy tiếp từ cell dưới.

In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("Không có GPU. Hãy chọn Runtime -> Change runtime type -> GPU")


PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [5]:
import os
import torch


from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    T5Tokenizer
)

from peft import (
    LoraConfig,
    PeftModel
)

# Lưu ý: SFTConfig thay thế cho TrainingArguments trong các bản TRL mới.
# dataset_text_field / max_length / packing giờ là thuộc tính của SFTConfig,
# không còn truyền trực tiếp vào SFTTrainer nữa.
from trl import SFTTrainer, SFTConfig


## Dataset tiếng Việt mẫu

Bạn có thể thay thế danh sách `data` bên dưới bằng dữ liệu của riêng bạn (càng nhiều mẫu càng tốt, tối thiểu vài trăm mẫu để có kết quả ổn).

In [6]:

datatrain_path = 'data/train'
datatest_path = 'data/test'

In [7]:
import os
import json

def load_data(dataset_path):
  dataset = []
  for filename in os.listdir(dataset_path):
      if filename.endswith('.json'):
          file_path = os.path.join(dataset_path, filename)

          with open(file_path, 'r', encoding='utf-8') as f:
              data = json.load(f)

          dataset.extend(data)
  return dataset

data_train = load_data(datatrain_path)
data_test = load_data(datatest_path)
# print(f"Tổng số mẫu: {len(dataset)}")
print(f"Tổng số mẫu train: {len(data_train)}")
print(f"Tổng số mẫu test: {len(data_test)}")


Tổng số mẫu train: 7461
Tổng số mẫu test: 60


In [8]:
from datasets import Dataset
# dataset = Dataset.from_list(dataset)
# dataset = dataset.train_test_split(
#     test_size=0.2,
#     seed=42
# )

train_dataset = Dataset.from_list(data_train)
test_dataset = Dataset.from_list(data_test)

# print(test_dataset)

print("Train:", len(train_dataset))
print("Test :", len(test_dataset))


Train: 7461
Test : 60


In [9]:
model_name = "VietAI/vit5-base"

import transformers
import tokenizers

print(transformers.__version__)
print(tokenizers.__version__)

print("Model:", model_name)


4.57.6
0.22.2
Model: VietAI/vit5-base


In [10]:
#module token
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Vocab size:", len(tokenizer))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Vocab size: 36096


In [11]:
def is_valid(example):
    return (
        example["input"] is not None
        and example["output"] is not None
        and str(example["input"]).strip() != ""
        and str(example["output"]).strip() != ""
    )

train_dataset = train_dataset.filter(is_valid)
test_dataset = test_dataset.filter(is_valid)

print(train_dataset)
print(test_dataset)

Filter:   0%|          | 0/7461 [00:00<?, ? examples/s]

Filter:   0%|          | 0/60 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output'],
    num_rows: 7452
})
Dataset({
    features: ['input', 'output'],
    num_rows: 59
})


In [12]:
def check_dataset(dataset, name):
    print(f"\n--- {name} ---")

    for col in dataset.column_names:
        invalid = []

        for i, value in enumerate(dataset[col]):
            if not isinstance(value, str) or not value.strip():
                invalid.append((i, repr(value)))
        print(f"Cột {col}: {len(invalid)} mẫu không hợp lệ")

check_dataset(train_dataset, "Train")
check_dataset(test_dataset, "Test")


--- Train ---
Cột input: 0 mẫu không hợp lệ
Cột output: 0 mẫu không hợp lệ

--- Test ---
Cột input: 0 mẫu không hợp lệ
Cột output: 0 mẫu không hợp lệ


In [13]:
import json

def format_target(output):
    output = output.replace("\n", "|||")
    return output


def preprocess(example):
    source = example["input"]
    target = example["output"]
    target = format_target(target)
    # if i <= 5:
    #   print("Dữ liệu sau khi format: ")
    #   print(target)
    model_inputs = tokenizer(
        source,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=target,
        max_length=512,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs
train_dataset = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)
test_dataset = test_dataset.map(preprocess, remove_columns=test_dataset.column_names)
print(train_dataset)


Map:   0%|          | 0/7452 [00:00<?, ? examples/s]

Map:   0%|          | 0/59 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7452
})


In [14]:
print(train_dataset['labels'])

Column([[2375, 238, 118, 212, 238, 183, 840, 1971, 868, 2792, 35790, 3430, 2035, 80, 655, 311, 305, 219, 171, 643, 868, 2792, 35790, 3430, 2035, 54, 1401, 387, 268, 4480, 1185, 38, 321, 1411, 2139, 2242, 35792, 2, 35882, 1306, 238, 118, 212, 238, 183, 271, 629, 943, 981, 821, 868, 2792, 80, 655, 311, 305, 219, 171, 643, 868, 2792, 35790, 3430, 2035, 54, 1401, 387, 268, 4480, 1185, 38, 321, 1411, 2139, 2242, 35792, 2, 35882, 1306, 238, 118, 212, 238, 183, 842, 1334, 943, 981, 821, 868, 2792, 80, 655, 311, 305, 219, 171, 643, 868, 2792, 35790, 3430, 2035, 54, 1401, 387, 268, 4480, 1185, 38, 321, 1411, 2139, 2242, 35792, 2, 35882, 1306, 238, 118, 212, 238, 183, 401, 814, 3430, 2035, 80, 655, 311, 305, 219, 171, 643, 868, 2792, 35790, 3430, 2035, 54, 1401, 387, 268, 4480, 1185, 38, 321, 1411, 2139, 2242, 35792, 2, 35882, 1306, 238, 118, 212, 238, 183, 489, 722, 362, 843, 3430, 2035, 80, 655, 311, 305, 219, 171, 643, 868, 2792, 35790, 3430, 2035, 54, 1401, 387, 268, 4480, 1185, 38, 321, 141

In [15]:
print(train_dataset)
output = tokenizer.decode(train_dataset['labels'][2])
print(output)

print(test_dataset)
output = tokenizer.decode(test_dataset['labels'][2])
print(output)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7452
})
Thi đua là hoạt động có tổ chức.<unk>Thi đua do người có thẩm quyền phát động.<unk>Thi đua có sự tham gia tự nguyện của cá nhân.<unk>Thi đua có sự tham gia tự nguyện của tập thể.</s>
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 59
})
Quyết định này quy định cách hiểu đối với các từ ngữ được sử dụng trong Quyết định.</s>


## Cấu hình QLoRA 4-bit và load model

In [16]:
# QUAN TRỌNG: dùng bfloat16 xuyên suốt (thay vì float16).
#
# Lý do: lỗi
#   NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda"
#   not implemented for 'BFloat16'
# xảy ra vì chế độ fp16=True bắt buộc dùng torch.cuda.amp.GradScaler, và
# GradScaler KHÔNG hỗ trợ bất kỳ tensor bfloat16 nào (Qwen2.5 vốn có vài
# phần luôn ở bfloat16 dù ta có ép torch_dtype=float16 hay không). Chỉ cần
# lọt một layer bfloat16 là toàn bộ training crash.
#
# Cách chắc chắn nhất: chuyển toàn bộ sang bfloat16. Khi dùng bf16=True,
# Trainer KHÔNG dùng GradScaler nữa -> lỗi này biến mất hoàn toàn.
# GPU T4 vẫn chạy được bfloat16 (không có Tensor Core native cho bf16 như
# GPU đời mới, nên hơi chậm hơn chút, nhưng chạy đúng và ổn định).

compute_dtype = torch.bfloat16
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     # NormalFloat4 - thường dùng trong QLoRA
#     bnb_4bit_quant_type="nf4",
#     # Double quantization giúp giảm memory thêm
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=compute_dtype
# )


In [17]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

print("Đã load model.")
print("Model dtype:", model.dtype)


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

Đã load model.
Model dtype: torch.bfloat16


In [18]:
# from peft import prepare_model_for_kbit_training

# # Bước chuẩn khi dùng QLoRA: đưa layernorm về fp32, bật input require_grad,
# # chuẩn bị model tương thích với gradient checkpointing. Thiếu bước này
# # cũng là nguyên nhân phổ biến gây lỗi dtype/NaN khi train.
# model = prepare_model_for_kbit_training(
#     model,
#     use_gradient_checkpointing=True
# )


In [19]:
# from peft import LoraConfig, get_peft_model

# lora_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     lora_dropout=0.05,
#     target_modules=["q", "k", "v", "o"],
#     bias="none",
#     task_type="SEQ_2_SEQ_LM"
# )

# model = get_peft_model(model, lora_config)

# model.print_trainable_parameters()

In [20]:
from transformers import Seq2SeqTrainingArguments

OUTPUT_DIR = "/content/drive/MyDrive/Project/Xet_Khen_Thuong/Fine-Tune-Vi-T5-Tach-Cau/Checkpoints"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer cho Full Fine-tuning
    optim="adamw_torch",

    lr_scheduler_type="cosine",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,

    logging_steps=1,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,

    predict_with_generate=True,

    fp16=False,
    bf16=True,

    gradient_checkpointing=True,

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


## Cấu hình training (`SFTConfig`)

**Lưu ý quan trọng:** trong các bản TRL mới (>=0.19–0.20 trở lên), `dataset_text_field`, `max_length` (đổi tên từ `max_seq_length`) và `packing` là thuộc tính của `SFTConfig`, không còn truyền trực tiếp vào `SFTTrainer` nữa. Đây là nguyên nhân phổ biến nhất gây lỗi `TypeError` khi chạy các đoạn code fine-tuning cũ.

In [21]:

# from transformers import Seq2SeqTrainingArguments

# OUTPUT_DIR = "/content/drive/MyDrive/Project/Xet_Khen_Thuong/Fine-Tune-Vi-T5-Tach-Cau/Checkpoints"

# training_args = Seq2SeqTrainingArguments(
#     output_dir=OUTPUT_DIR,

#     num_train_epochs=4,

#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=8,
#     optim="paged_adamw_8bit",

#     lr_scheduler_type="cosine",
#     learning_rate=1e-4,
#     weight_decay=0.01,

#     logging_steps=1,

#     eval_strategy="epoch",
#     save_strategy="epoch",
#     save_total_limit=3,

#     predict_with_generate=True,

#     fp16=False,
#     bf16=True,

#     report_to="none",

#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better=False,
# )

# from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer

# data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer,
#     model=model,
#     padding=True,
#     label_pad_token_id=-100,
#     return_tensors="pt"
# )

# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=test_dataset,
#     data_collator=data_collator
# )


In [22]:
import torch

batch = data_collator(
    [test_dataset[i] for i in range(2)]
)

batch = {
    k: v.to(model.device)
    for k, v in batch.items()
}

with torch.no_grad():
    outputs = model(**batch)

print("LOSS:", outputs.loss)
print("LOSS finite:", torch.isfinite(outputs.loss))

print("LOGITS finite:",
      torch.isfinite(outputs.logits).all())

LOSS: tensor(2.0312, device='cuda:0', dtype=torch.bfloat16)
LOSS finite: tensor(True, device='cuda:0')
LOGITS finite: tensor(True, device='cuda:0')


**KIỂM TRA BAD PARAMS**

In [23]:
import torch

bad_params = []

for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        bad_params.append(name)

print("Number of bad parameters:", len(bad_params))

for name in bad_params[:20]:
    print("BAD:", name)

Number of bad parameters: 0


In [24]:
# trainer.model.print_trainable_parameters()


In [25]:
sample = train_dataset[0]

print("input:", sample["input_ids"])
print("labels:", sample["labels"])

print("input length:", len(sample["input_ids"]))
print("label length:", len(sample["labels"]))

input: [252, 40, 35862, 1405, 448, 290, 2345, 2375, 238, 118, 212, 238, 183, 840, 1971, 35790, 271, 629, 35790, 842, 1334, 943, 981, 821, 868, 2792, 35790, 401, 814, 3430, 2035, 35899, 489, 722, 35790, 1107, 987, 362, 843, 3430, 2035, 35899, 2482, 868, 2792, 35790, 3430, 2035, 80, 655, 311, 305, 219, 171, 643, 868, 2792, 35790, 3430, 2035, 54, 1401, 387, 268, 4480, 1185, 38, 321, 1411, 2139, 2242, 35792, 1]
labels: [2375, 238, 118, 212, 238, 183, 840, 1971, 868, 2792, 35790, 3430, 2035, 80, 655, 311, 305, 219, 171, 643, 868, 2792, 35790, 3430, 2035, 54, 1401, 387, 268, 4480, 1185, 38, 321, 1411, 2139, 2242, 35792, 2, 35882, 1306, 238, 118, 212, 238, 183, 271, 629, 943, 981, 821, 868, 2792, 80, 655, 311, 305, 219, 171, 643, 868, 2792, 35790, 3430, 2035, 54, 1401, 387, 268, 4480, 1185, 38, 321, 1411, 2139, 2242, 35792, 2, 35882, 1306, 238, 118, 212, 238, 183, 842, 1334, 943, 981, 821, 868, 2792, 80, 655, 311, 305, 219, 171, 643, 868, 2792, 35790, 3430, 2035, 54, 1401, 387, 268, 4480, 118

In [26]:
print("\nTEST")
print(test_dataset[0])

for i in range(len(test_dataset)):
    labels = test_dataset[i]["labels"]

    print(
        i,
        "label length =", len(labels),
        "labels =", labels
    )


TEST
{'input_ids': [252, 40, 35792, 1405, 448, 290, 2345, 2375, 238, 118, 212, 238, 337, 1282, 1393, 145, 39, 1393, 163, 252, 60, 35885, 35790, 1393, 122, 5356, 60, 35890, 35790, 1393, 122, 252, 60, 35891, 35790, 1393, 122, 5356, 60, 35888, 35790, 1393, 163, 5356, 166, 35885, 35790, 1393, 60, 5356, 166, 35881, 39, 1393, 163, 5356, 151, 35885, 54, 1519, 10834, 50, 35890, 35856, 35852, 35846, 35852, 35852, 35856, 5382, 35839, 35881, 198, 40, 35881, 287, 163, 128, 60, 35846, 35852, 35852, 54, 161, 21861, 624, 19250, 2957, 2792, 35790, 3430, 2035, 35792, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [2375, 238, 118, 212, 238, 883, 1282, 1393, 145, 252, 60, 35885, 54, 1519, 210, 50, 35890, 35856, 35852, 35846, 35852, 35852, 

## Bắt đầu fine-tuning

Với dataset mẫu và cấu hình trên, quá trình này chạy trong vài phút trên GPU T4 miễn phí của Colab.

In [27]:
trainer.train()

# Sử dụng khi cần tiếp tục ở điểm trc
# Load check point
# import torch
# import numpy as np

# OUTPUT_DIR = (
#     "/content/drive/MyDrive/Project/Xet_Khen_Thuong/"
#     "Fine-Tune-Vi-T5-Tach-Cau/Checkpoints/checkpoint-2796/optimizer.pt"
# )

# torch.serialization.add_safe_globals([
#     np._core.multiarray._reconstruct,
#     np.ndarray
# ])

# checkpoint = torch.load(
#     OUTPUT_DIR,
#     map_location="cpu",
#     weights_only=True
# )

# print(type(checkpoint))
# print(checkpoint.keys() if isinstance(checkpoint, dict) else checkpoint)

Epoch,Training Loss,Validation Loss
1,0.218000,0.515426
2,0.291500,0.510979
3,0.574600,0.510991


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2796, training_loss=0.49852752220018826, metrics={'train_runtime': 4739.9966, 'train_samples_per_second': 4.716, 'train_steps_per_second': 0.59, 'total_flos': 2132393316480000.0, 'train_loss': 0.49852752220018826, 'epoch': 3.0})

In [28]:
# import torch
# import numpy as np

# safe_objects = [
#     np._core.multiarray._reconstruct,
#     np.ndarray,
#     np.dtype,
# ]

# # Cho phép các kiểu dtype NumPy đang gặp
# if hasattr(np, "dtypes"):
#     safe_objects.extend([
#         np.dtypes.Float64DType,
#         np.dtypes.UInt32DType,
#     ])

# torch.serialization.add_safe_globals(safe_objects)

# CHECKPOINT_PATH = (
#     "/content/drive/MyDrive/Project/Xet_Khen_Thuong/"
#     "Fine-Tune-Vi-T5-Tach-Cau/Checkpoints/checkpoint-2796"
# )

# trainer.train(resume_from_checkpoint=CHECKPOINT_PATH)

In [29]:
save_path = '/content/drive/MyDrive/Project/Xet_Khen_Thuong/Models/Vi-T5-Base-Tach-Cau'

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Đã lưu model tại:", save_path)


Đã lưu model tại: /content/drive/MyDrive/Project/Xet_Khen_Thuong/Models/Vi-T5-Base-Tach-Cau


## Kiểm tra model sau fine-tuning

In [30]:
# model.eval()
# question = """Cờ thi đua của Chính phủ”
# Khoản 2: Tập thể được xét tặng danh hiệu “Cờ thi đua của Chính phủ” theo quy định tại khoản 2 Điều 25 của Luật Thi đua, khen thưởng đạt các tiêu chuẩn:
# Điểm b) Thực hiện tốt chủ trương của Đảng, chính sách, pháp luật của Nhà nước xây dựng hệ thống chính trị trong sạch, vững mạnh;
# """

# # prompt = build_prompt(question)

# inputs = tokenizer(
#     question,
#     return_tensors="pt"
# ).to(model.device)

# with torch.no_grad():
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=500,
#         temperature=0.7,
#         top_p=0.9,
#         do_sample=True,
#         repetition_penalty=1.1,
#         pad_token_id=tokenizer.pad_token_id,
#     )

# answer = tokenizer.decode(
#     outputs[0][inputs["input_ids"].shape[1]:],
#     skip_special_tokens=True
# )

# print("Câu hỏi:")
# print(question)

# print("\nTrả lời:")
# print(answer)


In [31]:
# def chat(question: str) -> str:
#     prompt = build_prompt(question)

#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt"
#     ).to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=256,
#             temperature=0.7,
#             top_p=0.9,
#             do_sample=True,
#             repetition_penalty=1.1,
#             pad_token_id=tokenizer.pad_token_id,
#         )

#     response = tokenizer.decode(
#         outputs[0][inputs["input_ids"].shape[1]:],
#         skip_special_tokens=True
#     )

#     return response


In [32]:
# print(
#     chat(
#         "Tại sao Transformer phù hợp với xử lý ngôn ngữ tự nhiên?"
#     )
# )


In [33]:
# for i in range(10):
#     sample = train_dataset[i]

#     input_text = tokenizer.decode(
#         sample["input_ids"],
#         skip_special_tokens=True
#     )

#     print("INPUT:", input_text)

#     inputs = tokenizer(
#         input_text,
#         return_tensors="pt"
#     ).to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=128,
#             num_beams=4
#         )

#     prediction = tokenizer.decode(
#         outputs[0],
#         skip_special_tokens=True
#     )

#     target = tokenizer.decode(
#         sample["labels"],
#         skip_special_tokens=True
#     )

#     print("PRED :", prediction)
#     print("TRUE :", target)
#     print("-" * 80)

In [34]:
# for i in range(10):
#     sample = test_dataset[i]

#     input_text = tokenizer.decode(
#         sample["input_ids"],
#         skip_special_tokens=True
#     )

#     print("INPUT:", input_text)

#     inputs = tokenizer(
#         input_text,
#         return_tensors="pt"
#     ).to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=128,
#             num_beams=4
#         )

#     prediction = tokenizer.decode(
#         outputs[0],
#         skip_special_tokens=True
#     )

#     target = tokenizer.decode(
#         sample["labels"],
#         skip_special_tokens=True
#     )

#     print("PRED :", prediction)
#     print("TRUE :", target)
#     print("-" * 80)

In [35]:
# processed = preprocess(train_dataset)

# print("INPUT:")
# print(tokenizer.decode(
#     processed["input_ids"],
#     skip_special_tokens=False
# ))

# print("\nLABEL:")
# print(tokenizer.decode(
#     processed["labels"],
#     skip_special_tokens=False
# ))

#XÀI LOCAL LLM CHO RAG

In [36]:
# USE_LOCAL_MODEL = True  # đổi thành False nếu muốn dùng Gemini API thay vì model cục bộ

In [37]:
# if USE_LOCAL_MODEL:
#     import os
#     import torch
#     from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
#     from peft import PeftModel

#     LOCAL_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
#     LOCAL_ADAPTER_PATH = "qwen-vietnamese-lora-final"  # đổi nếu bạn đặt tên thư mục khác

#     # Dùng bfloat16 xuyên suốt để tránh lỗi GradScaler/BFloat16 thường gặp khi
#     # trộn lẫn fp16 và bf16 (lưu ý: đây là suy luận/inference nên không cần
#     # GradScaler, nhưng vẫn nên đồng nhất dtype để tránh cảnh báo/lỗi không cần thiết).
#     compute_dtype = torch.bfloat16

#     bnb_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_quant_type="nf4",
#         bnb_4bit_use_double_quant=True,
#         bnb_4bit_compute_dtype=compute_dtype,
#     )

#     local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME, trust_remote_code=True)
#     if local_tokenizer.pad_token is None:
#         local_tokenizer.pad_token = local_tokenizer.eos_token

#     print(f"[Local LLM] Đang tải model gốc: {LOCAL_MODEL_NAME} ...")
#     local_base_model = AutoModelForCausalLM.from_pretrained(
#         LOCAL_MODEL_NAME,
#         quantization_config=bnb_config,
#         torch_dtype=compute_dtype,
#         device_map="auto",
#         trust_remote_code=True,
#     )
#     # Đây là suy luận (inference), không phải training, nên bật lại cache để sinh
#     # văn bản nhanh hơn (khác với lúc fine-tune phải tắt use_cache).
#     local_base_model.config.use_cache = True

#     if os.path.isdir(LOCAL_ADAPTER_PATH):
#         print(f"[Local LLM] Tìm thấy adapter LoRA tại '{LOCAL_ADAPTER_PATH}', đang gắn vào model gốc...")
#         local_model = PeftModel.from_pretrained(local_base_model, LOCAL_ADAPTER_PATH)
#     else:
#         print(
#             f"[Local LLM] Không thấy thư mục adapter '{LOCAL_ADAPTER_PATH}'. "
#             "Dùng thẳng model Qwen2.5-1.5B-Instruct gốc (khuyến nghị cho RAG)."
#         )
#         local_model = local_base_model

#     local_model.eval()
#     print("[Local LLM] Sẵn sàng.")
# else:
#     local_model = None
#     local_tokenizer = None


In [38]:
# import torch

# LOCAL_SYSTEM_PROMPT = (
#     "Bạn là trợ lý AI sử dụng tiếng Việt. "
#     "Hãy trả lời chính xác, rõ ràng và dễ hiểu."
# )


# def local_generate(prompt, max_new_tokens=400, temperature=0.3):
#     """Sinh câu trả lời bằng model Qwen2.5 cục bộ (dùng khi USE_LOCAL_MODEL=True)."""
#     messages = [
#         {"role": "system", "content": LOCAL_SYSTEM_PROMPT},
#         {"role": "user", "content": prompt},
#     ]
#     text = local_tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     inputs = local_tokenizer(text, return_tensors="pt").to(local_model.device)

#     with torch.no_grad():
#         outputs = local_model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             temperature=temperature,
#             top_p=0.9,
#             do_sample=True,
#             repetition_penalty=1.1,
#             pad_token_id=local_tokenizer.pad_token_id,
#         )

#     return local_tokenizer.decode(
#         outputs[0][inputs["input_ids"].shape[1]:],
#         skip_special_tokens=True,
#     )


# print("[Local LLM] Đã định nghĩa hàm local_generate().")

In [39]:
# import os
# from google.colab import files

# os.makedirs("documents", exist_ok=True)
# uploaded = files.upload()

# for fname in uploaded.keys():
#     dest = os.path.join("documents", fname)
#     os.rename(fname, dest)

# print("\nĐã tải lên và lưu vào thư mục 'documents/':")
# for fname in uploaded.keys():
#     print(" -", fname)

In [40]:
# !pip install -q sentence-transformers faiss-cpu

In [41]:
# print(local_generate("Xin chào. Hãy trả lời một câu ngắn bằng tiếng Việt."))

In [42]:
# import glob
# import numpy as np
# from sentence_transformers import SentenceTransformer
# import faiss

# try:
#     from pypdf import PdfReader
# except ImportError:
#     PdfReader = None

# # ---------------- Cấu hình ----------------
# # Model embedding chạy CỤC BỘ, MIỄN PHÍ (đa ngôn ngữ, hỗ trợ tốt tiếng Việt).
# EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# DOCUMENTS_FOLDER = "/content/documents"
# CHUNK_SIZE = 500      # số từ mỗi đoạn
# CHUNK_OVERLAP = 80    # số từ chồng lấn giữa 2 đoạn liên tiếp
# TOP_K = 3             # số đoạn liên quan nhất lấy ra để hỏi LLM


# # ---------------- Đọc dữ liệu ----------------
# def read_txt_file(path):
#     with open(path, "r", encoding="utf-8", errors="ignore") as f:
#         return f.read()

# def read_pdf_file(path):
#     if PdfReader is None:
#         raise RuntimeError("Chưa cài pypdf. Chạy: !pip install pypdf")
#     reader = PdfReader(path)
#     return "\n".join(page.extract_text() or "" for page in reader.pages)

# def load_documents(folder):
#     docs = []
#     paths = glob.glob(os.path.join(folder, "**", "*.*"), recursive=True)
#     for path in paths:
#         ext = os.path.splitext(path)[1].lower()
#         try:
#             if ext in (".txt", ".md"):
#                 text = read_txt_file(path)
#             elif ext == ".pdf":
#                 text = read_pdf_file(path)
#             else:
#                 continue
#             if text.strip():
#                 docs.append({"source": os.path.basename(path), "text": text})
#         except Exception as e:
#             print(f"[Cảnh báo] Không đọc được file {path}: {e}")
#     return docs


# # ---------------- Chia đoạn (chunking) ----------------
# def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
#     words = text.split()
#     if not words:
#         return []
#     chunks = []
#     step = max(chunk_size - overlap, 1)
#     for start in range(0, len(words), step):
#         chunk_words = words[start:start + chunk_size]
#         if chunk_words:
#             chunks.append(" ".join(chunk_words))
#         if start + chunk_size >= len(words):
#             break
#     return chunks


# # ---------------- Pipeline RAG ----------------
# class RAGPipeline:
#     def __init__(self, embedding_model_name=EMBEDDING_MODEL_NAME):
#         print(f"[Khởi tạo] Đang tải model embedding cục bộ: {embedding_model_name} ...")
#         self.embedder = SentenceTransformer(embedding_model_name)
#         self.index = None
#         self.chunk_texts = []
#         self.chunk_sources = []

#     def build_index(self, folder):
#         docs = load_documents(folder)
#         if not docs:
#             raise RuntimeError(
#                 f"Không tìm thấy file .txt/.pdf nào trong thư mục '{folder}'. "
#                 f"Hãy chạy lại Bước 3 để tải tài liệu lên."
#             )

#         print(f"[Nạp dữ liệu] Đọc được {len(docs)} tài liệu. Đang chia đoạn (chunking)...")
#         for doc in docs:
#             for chunk in chunk_text(doc["text"]):
#                 self.chunk_texts.append(chunk)
#                 self.chunk_sources.append(doc["source"])

#         print(f"[Nạp dữ liệu] Tổng cộng {len(self.chunk_texts)} đoạn (chunks).")
#         print("[Embedding] Đang tính vector cho từng đoạn (chạy cục bộ, miễn phí)...")

#         embeddings = self.embedder.encode(
#             self.chunk_texts,
#             batch_size=32,
#             show_progress_bar=True,
#             normalize_embeddings=True,
#         )
#         embeddings = np.asarray(embeddings, dtype="float32")

#         dim = embeddings.shape[1]
#         self.index = faiss.IndexFlatIP(dim)
#         self.index.add(embeddings)

#         print("[Hoàn tất] Chỉ mục FAISS đã sẵn sàng.")

#     def retrieve(self, query, top_k=TOP_K):
#         query_vec = self.embedder.encode([query], normalize_embeddings=True)
#         query_vec = np.asarray(query_vec, dtype="float32")
#         scores, indices = self.index.search(query_vec, top_k)
#         results = []
#         for score, idx in zip(scores[0], indices[0]):
#             if idx == -1:
#                 continue
#             results.append({
#                 "text": self.chunk_texts[idx],
#                 "source": self.chunk_sources[idx],
#                 "score": float(score),
#             })
#         return results

#     def generate_answer(self, query, contexts):
#         context_block = "\n\n".join(
#             f"[Nguon: {c['source']}]\n{c['text']}" for c in contexts
#         )
#         prompt = f"""Bạn là một trợ lý trả lời câu hỏi dựa HOÀN TOÀN trên các đoạn
# văn bản ngữ cảnh được cung cấp bên dưới. Nếu ngữ cảnh không chứa đủ thông tin
# để trả lời, hãy nói rõ là bạn không tìm thấy thông tin đó trong tài liệu,
# KHÔNG được tự bịa ra thông tin.

# ### NGỮ CẢNH:
# {context_block}

# ### CÂU HỎI:
# {query}

# ### TRẢ LỜI (ngắn gọn, chính xác, dựa trên ngữ cảnh trên):"""

#         # Model 1.5B khá nhỏ nên giữ prompt ngắn gọn, temperature thấp để bám
#         # sát ngữ cảnh, hạn chế bịa thông tin.
#         return local_generate(prompt, max_new_tokens=400, temperature=0.3)

#     def ask(self, query, top_k=TOP_K):
#         contexts = self.retrieve(query, top_k=top_k)
#         if not contexts:
#             return "Không tìm thấy đoạn văn bản nào liên quan trong tài liệu.", []
#         answer = self.generate_answer(query, contexts)
#         return answer, contexts

# print("Đã định nghĩa xong pipeline RAGPipeline.")


In [43]:
# rag = RAGPipeline()
# rag.build_index(DOCUMENTS_FOLDER)

In [44]:
# query = "What is AIM-MD?"

# answer, contexts = rag.ask(query)

# print("--- TRẢ LỜI ---")
# print(answer)

# print("\n--- NGUỒN THAM KHẢO (retrieval) ---")
# for i, c in enumerate(contexts, 1):
#     preview = c["text"][:150].replace("\n", " ")
#     print(f"[{i}] {c['source']} (độ tương đồng={c['score']:.3f}): {preview}...")
